# genai-corpus smoke test

CI's notebook smoke test: import the package, load the committed fixture
corpus, and verify every fixture asset's checksum. Runs on CPU against a clean clone,
with no credential, no network and no GPU.

In [ ]:
from pathlib import Path

import genai_corpus
from genai_corpus import load_manifest, verify_checksums

print(f"genai_corpus version: {genai_corpus.__version__}")

In [ ]:
# Anchor on the installed package's own file rather than the kernel's cwd: the
# latter is notebooks/ under CI and Jupyter Lab, but not guaranteed if the kernel
# starts at the repo root instead.
REPO_ROOT = Path(genai_corpus.__file__).resolve().parent.parent.parent
FIXTURE_ROOT = REPO_ROOT / "fixtures" / "corpus"
MANIFEST = FIXTURE_ROOT / "manifest.json"

assets = load_manifest(MANIFEST)
print(f"loaded {len(assets)} fixture assets")
assert len(assets) > 0, "fixture manifest is empty"

In [ ]:
mismatches = verify_checksums(MANIFEST, FIXTURE_ROOT)
print(f"checksum mismatches: {mismatches}")
assert mismatches == [], f"fixture corpus checksum mismatch: {mismatches}"

In [ ]:
total_bytes = sum(p.stat().st_size for p in FIXTURE_ROOT.rglob("*") if p.is_file())
print(f"fixture corpus size: {total_bytes} bytes")
assert total_bytes < 5 * 1024 * 1024, "fixture corpus exceeds the 5 MB cap"
print("smoke test passed")